<a href="https://colab.research.google.com/github/officiallong/BUS4-118S-build-ai-agents-and-chatbots-with-langgraph-2021112/blob/main/Integrated_Laptop_Chatbot_Backend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laptop Customer Service — Integrated Multi-Agent Chatbot

This notebook integrates the Product Recommendation Agent, Order Status Agent, and Refund Agent into a single LangGraph-routed customer service chatbot.

## Section 1 — Install Required Libraries

In [19]:
%pip install -q -U "langchain>=1,<2" "langgraph>=1,<2" "langchain-google-genai>=4,<5" langchain-community langchain-text-splitters pypdf pandas

## Section 2 — Configure Gemini API and Shared Language Model

In [20]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

api_key = userdata.get("GEMINI_API_KEY")
if not api_key:
    raise ValueError("Add GEMINI_API_KEY to Colab Secrets and grant notebook access.")
os.environ["GOOGLE_API_KEY"] = api_key

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=api_key,
    temperature=0
)

embedding = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=api_key
)

print("Shared Gemini model and embedding model configured.")

Shared Gemini model and embedding model configured.


## Section 3 — Load Product and Order Data
**Product teammate:** Download the laptop pricing CSV and product-description PDF from the same GitHub files used in the Product notebook.

**Order teammate:** This notebook automatically creates a **fictional sample** `Laptop Orders.csv` if an original file is not already present at `/content/Laptop Orders.csv`. To use the teammate's original data, upload it to Colab Files with that exact filename **before** running the order-data cell. No manual upload is required for the sample demo.


In [21]:
!wget -q -O "/content/Laptop pricing.csv" \
"https://raw.githubusercontent.com/LinkedInLearning/build-ai-agents-and-chatbots-with-langgraph-2021112/main/data/Laptop%20pricing.csv"

!wget -q -O "/content/Laptop product descriptions.pdf" \
"https://raw.githubusercontent.com/LinkedInLearning/build-ai-agents-and-chatbots-with-langgraph-2021112/main/data/Laptop%20product%20descriptions.pdf"

!ls -lh "/content/Laptop pricing.csv" \
"/content/Laptop product descriptions.pdf"
from pathlib import Path
for path in ["/content/Laptop pricing.csv", "/content/Laptop product descriptions.pdf"]:
    if not Path(path).exists() or Path(path).stat().st_size == 0:
        raise FileNotFoundError(f"Missing or empty product file: {path}")
# Check file signatures so an HTML error page isn't mistaken for a PDF.
if Path("/content/Laptop product descriptions.pdf").read_bytes()[:4] != b"%PDF":
    raise ValueError("Product PDF download is invalid. Check the source URL.")
print("Product data downloaded.")

-rw-r--r-- 1 root root  132 Oct 20 06:45 '/content/Laptop pricing.csv'
-rw-r--r-- 1 root root 1.9K Oct 20 06:45 '/content/Laptop product descriptions.pdf'
Product data downloaded.


In [22]:
from pathlib import Path
import pandas as pd

order_path = Path("/content/Laptop Orders.csv")

if order_path.is_file():
    print("Using existing Laptop Orders.csv from this Colab runtime.")
else:
    # Fictional classroom demonstration records; NOT real customer orders.
    sample_orders = [
        {"Order ID": "ORD-7311", "Product": "AlphaBook Pro", "Quantity Ordered": 2, "Delivery Date": "2026-10-10", "Status": "Delivered"},
        {"Order ID": "ORD-7312", "Product": "GammaAir X", "Quantity Ordered": 1, "Delivery Date": "2026-10-23", "Status": "Shipped"},
        {"Order ID": "ORD-7313", "Product": "SpectraBook S", "Quantity Ordered": 1, "Delivery Date": "2026-10-28", "Status": "Processing"},
        {"Order ID": "ORD-7314", "Product": "OmegaPro G17", "Quantity Ordered": 1, "Delivery Date": "2026-11-02", "Status": "Processing"},
        {"Order ID": "ORD-7315", "Product": "NanoEdge Flex", "Quantity Ordered": 2, "Delivery Date": "2026-10-18", "Status": "Delivered"},
    ]
    pd.DataFrame(sample_orders).to_csv(order_path, index=False)
    print("Created fictional sample order data at /content/Laptop Orders.csv.")

product_orders_df = pd.read_csv(order_path)
required_order_columns = {"Order ID", "Quantity Ordered"}
missing = required_order_columns - set(product_orders_df.columns)
if missing:
    raise ValueError(f"Order CSV is missing required columns: {sorted(missing)}")

print(f"Order data loaded: {len(product_orders_df)} records")
display(product_orders_df.head())
# Validate order data before any agent tools use it.
if product_orders_df.empty:
    raise ValueError("Order dataset is empty.")
if product_orders_df["Order ID"].isna().any() or product_orders_df["Order ID"].duplicated().any():
    raise ValueError("Order IDs must be present and unique.")
if pd.to_numeric(product_orders_df["Quantity Ordered"], errors="coerce").isna().any():
    raise ValueError("Order quantities must be numeric.")


Using existing Laptop Orders.csv from this Colab runtime.
Order data loaded: 5 records


,Order ID,Product,Quantity Ordered,Delivery Date,Status
0,ORD-7311,AlphaBook Pro,2,2026-10-10,Delivered
1,ORD-7312,GammaAir X,1,2026-10-23,Shipped
2,ORD-7313,SpectraBook S,1,2026-10-28,Processing
3,ORD-7314,OmegaPro G17,1,2026-11-02,Processing
4,ORD-7315,NanoEdge Flex,2,2026-10-18,Delivered


## Section 4 — Product Recommendation Agent (Product Teammate)
Includes laptop price lookup, available inventory, PDF specification retrieval, and the agent system prompt. The parent Main Chatbot maintains conversation memory, so this subagent does not use its own checkpointer.

In [23]:
import pandas as pd
from langchain_core.tools import tool

product_pricing_df = pd.read_csv("/content/Laptop pricing.csv")

print(product_pricing_df)


@tool
def get_laptop_price(laptop_name: str) -> int:
    """
    Returns the price of a laptop.
    Returns -1 if the laptop is not found.
    """
    matches = product_pricing_df[
        product_pricing_df["Name"].str.contains(
            laptop_name,
            case=False,
            na=False
        )
    ]

    if len(matches) == 0:
        return -1

    return int(matches["Price"].iloc[0])


@tool
def list_available_laptops() -> str:
    """
    Returns all available laptops, prices, and shipping times.
    """
    return product_pricing_df.to_string(index=False)


print("Pricing tools created successfully!")

            Name  Price  ShippingDays
0  AlphaBook Pro   1499             2
1     GammaAir X   1399             7
2  SpectraBook S   2499             7
3   OmegaPro G17   2199            14
4  NanoEdge Flex   1699             2
Pricing tools created successfully!


In [24]:
from langchain_core.tools import create_retriever_tool
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(
    "/content/Laptop product descriptions.pdf"
)

docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1024,
    chunk_overlap=256
)

splits = text_splitter.split_documents(docs)

prod_feature_store = InMemoryVectorStore.from_documents(
    documents=splits,
    embedding=embedding
)

get_product_features = create_retriever_tool(
    prod_feature_store.as_retriever(
        search_kwargs={"k": 2}
    ),
    name="Get_Product_Features",
    description="""
    This tool contains laptop information, including processors,
    memory, storage, design, and advantages.
    Use this tool to find laptop features.
    """
)

print("Product features tool created successfully!")

Product features tool created successfully!


In [25]:
from langchain.agents import create_agent

recommendation_system_prompt = """
You are a helpful Product Recommendation Agent for an online laptop store.

Your job is to recommend laptops based on the customer's:
- Budget
- Intended use
- Performance needs
- Portability preferences

Rules:
1. Use the available tools to check prices and features.
2. Never invent laptop information.
3. Ask follow-up questions when necessary.
4. Recommend one or two laptops.
5. Explain why each recommendation fits the customer.
6. Clearly state the laptop name and price.
7. Be friendly and professional.
"""

recommendation_tools = [
    list_available_laptops,
    get_laptop_price,
    get_product_features
]


product_recommendation_agent = create_agent(
    model=llm,
    tools=recommendation_tools,
    system_prompt=recommendation_system_prompt
)

print("Product Recommendation Agent created successfully!")

Product Recommendation Agent created successfully!


## Section 5 — Order Status Agent (Order Teammate)

In [26]:
from langchain_core.tools import tool

@tool
def get_order_details(order_id: str) -> str:
    """
    This function returns details about a laptop order, given an order ID
    It performs an exact match between the input order id and available order ids
    If a match is found, it returns products (laptops) ordered, quantity ordered and delivery date.
    If there is NO match found, it returns -1
    """
    #Filter Dataframe for order ID
    match_order_df = product_orders_df[
                        product_orders_df["Order ID"] == order_id ]

    #Check if a record was found, if not return -1
    if len(match_order_df) == 0 :
        return f"Order {order_id} not found."
    else:
        return str(match_order_df.iloc[0].to_dict())

#Test the tool. Before running the test, comment the @tool annotation
#print(get_order_details("ORD-6948"))
#print(get_order_details("ORD-9999"))

@tool
def update_quantity(order_id: str, new_quantity: int) -> str:
    """
    This function updates the quantity of products ( laptops ) ordered for a given order Id.
    It there are no matching orders, it returns False.
    """
    if new_quantity < 1:
        return "Quantity must be at least 1."
    #Find if matching record exists
    match_order_df = product_orders_df[
                        product_orders_df["Order ID"] == order_id ]

    #Check if a record was found, if not return -1
    if len(match_order_df) == 0 :
        return f"Order {order_id} not found."
    else:
        product_orders_df.loc[
            product_orders_df["Order ID"] == order_id,
                "Quantity Ordered"] = new_quantity
        return f"Updated order {order_id} to quantity {new_quantity} in this session."

#Test the tool. Before running the test, comment the @tool annotation
#print(get_order_details("ORD-6948"))
#print(update_quantity("ORD-6948", 1))
#print(get_order_details("ORD-6948"))
#print(update_quantity("ORD-9999",2))
#print(product_orders_df)

In [27]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
import json

#An Agent State class that keep state of the agent while it answers a query
class OrdersAgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

#-----------------------------------------------------------------------------
#An agent class that manages all agentic interactions
class OrdersAgent:

    #Setup the agent graph, tools and memory
    def __init__(self, model, tools, system_prompt, debug):

        self.system_prompt=system_prompt
        self.debug=debug

        #Setup the graph for the agent manually
        agent_graph=StateGraph(OrdersAgentState)
        agent_graph.add_node("orders_llm",self.call_llm)
        agent_graph.add_node("orders_tools",self.call_tools)
        agent_graph.add_conditional_edges(
            "orders_llm",
            self.is_tool_call,
            {True: "orders_tools", False: END }
        )
        agent_graph.add_edge("orders_tools","orders_llm")
        #Set where there graph starts
        agent_graph.set_entry_point("orders_llm")

        #Add chat memory
        #compile the graph
        self.agent_graph = agent_graph.compile()

        #Setup tools
        self.tools = { tool.name : tool for tool in tools }
        if self.debug:
            print("\nTools loaded :", self.tools)

        #attach tools to model
        self.model=model.bind_tools(tools)


    #Call the LLM with the messages to get next action/result
    def call_llm(self, state:OrdersAgentState):

        messages=state["messages"]

        #If system prompt exists, add to messages in the front
        if self.system_prompt:
            messages = [SystemMessage(content=self.system_prompt)] + messages

        #invoke the model with the message history
        result = self.model.invoke(messages)
        if self.debug:
            print(f"\nLLM Returned : {result}")
        #Return the LLM output
        return { "messages":[result] }


    #Check if the next action is a tool call.
    def is_tool_call(self, state:OrdersAgentState):
        last_message = state["messages"][-1]
        #print("Last result from LLM : ", last_message)
        #If tool action is requested
        if len(getattr(last_message, "tool_calls", [])) > 0 :
            return True
        else:
            return False

    #Execute the tool requested with the given parameters
    def call_tools(self, state:OrdersAgentState):
        #Get last message
        tool_calls = state["messages"][-1].tool_calls
        results=[]

        #Multiple tool calls may be requested. Execute one by one
        for tool in tool_calls:
            #Handle tool missing error
            if not tool["name"] in self.tools:
                print(f"Unknown tool name {tool}")
                result = "Invalid tool found. Please retry"
            else:
                #Call the tool and collect results
                result=self.tools[tool["name"]].invoke(tool["args"])

            #append results to the list of tool results
            results.append(ToolMessage(tool_call_id=tool['id'],
                                       name=tool['name'],
                                       content=str(result)))

            if self.debug:
                print(f"\nTools returned {results}")
        # Return all tool results after processing every requested call.
        return { "messages" : results }

#-----------------------------------------------------------------------------
#Setup the custom agent

#Note that this is a string, since the model init only accepts a string.
order_system_prompt = """
    You are professional chatbot that manages orders for laptops sold by our company.
    The tools allow for retrieving order details as well as update order quantity.
    Do NOT reveal information about other orders than the one requested.
    You will handle small talk and greetings by producing professional responses.
    """

#Create the custom orders agent
orders_agent = OrdersAgent(llm,
                           [get_order_details, update_quantity],
                           order_system_prompt,
                           debug=False)

#Visualize the Agent
print("Order Status Agent created successfully!")


Order Status Agent created successfully!


## Section 6 — Refund Agent

In [28]:
from langchain.agents import create_agent
from langchain_core.tools import tool

SAMPLE_RETURN_DAYS = 30


@tool
def check_refund_policy() -> str:
    """Look up the sample laptop store refund policy, return window, and refund procedure."""
    return (
        "SAMPLE policy: Returns within 30 days of delivery; "
        "laptop must be in good condition; "
        "proof of purchase required; "
        "refunds use the original payment method. "
        "Contact customer support to request a return."
    )


@tool
def check_refund_eligibility(
    days_since_delivery: int,
    good_condition: bool,
    has_proof_of_purchase: bool
) -> str:
    """Check sample refund eligibility using days since delivery, laptop condition, and receipt. Never guess missing information."""

    if days_since_delivery < 0:
        return "Invalid days_since_delivery; ask the customer to clarify."

    problems = []

    if days_since_delivery > SAMPLE_RETURN_DAYS:
        problems.append("more than 30 days since delivery")

    if not good_condition:
        problems.append("laptop not in good condition")

    if not has_proof_of_purchase:
        problems.append("proof of purchase missing")

    if problems:
        return (
            "Not eligible under SAMPLE policy: "
            + "; ".join(problems)
        )

    return (
        "Meets SAMPLE eligibility criteria. "
        "Contact support to request a return; "
        "no refund has been approved or processed."
    )


agent_tools = [
    check_refund_policy,
    check_refund_eligibility
]


system_prompt = """
You are a laptop store Refund Agent for a classroom demo.

All refund rules are SAMPLE policies, not a real store's official policy.

Use your refund tools to answer policy questions and check hypothetical eligibility.

Never invent policy terms or claim a refund was approved, issued, or processed.

Ask for missing eligibility details instead of guessing them.

Be clear, polite, and concise. Only handle refunds and returns.
"""

# Refund Agent — Actual
refund_graph = create_agent(
    model=llm,
    tools=agent_tools,
    system_prompt=system_prompt
)
print("Refund Agent created successfully!")


Refund Agent created successfully!


## Section 7 — Building and Connecting the Main Chatbot with LangGraph

In [36]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, AIMessage


# Shared conversation state
class ChatState(MessagesState):
    route: str


# Instructions for the routing agent
ROUTER_PROMPT = """You are a routing agent for a laptop customer service chatbot.

Choose exactly ONE of these labels:
product, order, refund, clarify

ROUTING RULES:
- product: Laptop recommendations, prices, specifications,
  availability, comparisons, budgets, or preferences.
- order: Order status, tracking, delivery dates,
  order IDs, or quantity changes.
- refund: Refund policies, returns, eligibility,
  receipts, or refund requests.
- clarify: Greetings or questions unrelated to these services.

IMPORTANT:
- Always consider the conversation history.
- If the customer is continuing an earlier conversation,
  keep routing to the same agent.
- For example, "My budget is $1,500" and
  "I prefer something lightweight" are product follow-ups.
- If the customer clearly switches topics,
  route to the newly requested agent.
- Respond with ONLY the lowercase label.
"""


# Router determines which specialized agent to use
def router(state: ChatState):

    valid_routes = {"product", "order", "refund", "clarify"}

    result = llm.invoke(
        [{"role": "system", "content": ROUTER_PROMPT}]
        + state["messages"]
    )

    content = result.content

    # Handle Gemini text-block responses
    if isinstance(content, list):
        content = " ".join(
            block.get("text", "")
            for block in content
            if isinstance(block, dict)
        )

    choice = str(content).strip().lower()

    # Normalize unexpected model output
    if choice not in valid_routes:
        for label in ("product", "order", "refund", "clarify"):
            if choice.startswith(label):
                choice = label
                break
        else:
            choice = "clarify"

    # Keep the previous agent for ambiguous follow-up messages
    if choice == "clarify":
        previous_route = state.get("route")

        if previous_route in {"product", "order", "refund"}:
            choice = previous_route

    return {"route": choice}


# Extract the final response from a specialized agent
def _final_answer(result):
    last = result["messages"][-1]
    return AIMessage(content=last.content)


# Product Recommendation Agent
def product_node(state: ChatState):
    result = product_recommendation_agent.invoke(
        {"messages": state["messages"]}
    )
    return {"messages": [_final_answer(result)]}


# Order Status Agent
def order_node(state: ChatState):
    result = orders_agent.agent_graph.invoke(
        {"messages": state["messages"]}
    )
    return {"messages": [_final_answer(result)]}


# Refund Policy Agent
def refund_node(state: ChatState):
    result = refund_graph.invoke(
        {"messages": state["messages"]}
    )
    return {"messages": [_final_answer(result)]}


# Clarification Agent
def clarify_node(state: ChatState):
    return {
        "messages": [
            AIMessage(
                content=(
                    "Hi! I can help with laptop recommendations, "
                    "order status, or refunds. "
                    "What would you like help with?"
                )
            )
        ]
    }


# Build the main LangGraph workflow
builder = StateGraph(ChatState)

builder.add_node("router", router)
builder.add_node("product", product_node)
builder.add_node("order", order_node)
builder.add_node("refund", refund_node)
builder.add_node("clarify", clarify_node)

builder.add_edge(START, "router")

builder.add_conditional_edges(
    "router",
    lambda state: state["route"],
    {
        "product": "product",
        "order": "order",
        "refund": "refund",
        "clarify": "clarify"
    }
)

for name in ("product", "order", "refund", "clarify"):
    builder.add_edge(name, END)


# Shared conversation memory
main_chatbot = builder.compile(
    checkpointer=InMemorySaver()
)

print(
    "Main Chatbot compiled: "
    "Product + Order + Refund + shared memory."
)


Main Chatbot compiled: Product + Order + Refund + shared memory.


In [30]:
assert {"Order ID", "Quantity Ordered"}.issubset(product_orders_df.columns)
assert len(product_orders_df) >= 1
assert product_recommendation_agent is not None
assert orders_agent.agent_graph is not None
assert refund_graph is not None
assert main_chatbot is not None

# Test the tools directly without using Gemini.
check_id = str(product_orders_df["Order ID"].iloc[0])
original_qty = int(product_orders_df.loc[product_orders_df["Order ID"] == check_id, "Quantity Ordered"].iloc[0])
assert check_id in get_order_details.invoke({"order_id": check_id})
assert "not found" in get_order_details.invoke({"order_id": "ORD-DOES-NOT-EXIST"}).lower()
assert "at least 1" in update_quantity.invoke({"order_id": check_id, "new_quantity": 0})
assert "Updated" in update_quantity.invoke({"order_id": check_id, "new_quantity": original_qty + 1})
assert int(product_orders_df.loc[product_orders_df["Order ID"] == check_id, "Quantity Ordered"].iloc[0]) == original_qty + 1
# Restore the original quantity so demo tests start with the same data.
update_quantity.invoke({"order_id": check_id, "new_quantity": original_qty})
print("Offline integration checks passed. Next, run the Gemini live tests below.")

Offline integration checks passed. Next, run the Gemini live tests below.


## Section 8 — Testing All Three Agents

In [31]:
import uuid

def ask_main(question, thread_id=None):
    thread_id = thread_id or str(uuid.uuid4())
    result = main_chatbot.invoke(
        {"messages": [HumanMessage(content=question)]},
        config={"configurable": {"thread_id": thread_id}}
    )
    answer = result["messages"][-1].content
    print(f"USER: {question}\nBOT: {answer}\n")
    return answer

ask_main("Can you recommend a laptop for college under $1,500?")
example_order_id = str(product_orders_df["Order ID"].iloc[0])
ask_main(f"What is the status of order {example_order_id}?")
ask_main("Can I return a laptop 15 days after delivery if it's in good condition and I have the receipt?")

USER: Can you recommend a laptop for college under $1,500?
BOT: [{'type': 'text', 'text': 'Here are two excellent laptop recommendations for college that fit within your $1,500 budget:\n\n---\n\n### 1. **GammaAir X** — **$1,399**\n* **Key Specifications:** AMD Ryzen 7 processor, 32GB DDR4 RAM, 512GB NVMe SSD\n* **Why it’s great for college:** \n  * **Exceptional Multitasking:** With 32GB of RAM, it can easily handle dozens of browser tabs, virtual lectures, coding environments, or data analysis tools without slowing down.\n  * **Portability:** It features a thin and light form factor, making it comfortable to carry between dorms, libraries, and lecture halls.\n\n---\n\n### 2. **AlphaBook Pro** — **$1,499**\n* **Key Specifications:** 12th Gen Intel Core i7 processor, 16GB DDR4 RAM, 1TB SSD\n* **Why it’s great for college:**\n  * **Ample Storage:** The 1TB SSD gives you plenty of space to store large files, course materials, software, and personal media over your entire college career.\n

[{'type': 'text',
  'text': 'Based on our sample policy, returning a laptop 15 days after delivery in good condition with proof of purchase meets the sample eligibility criteria. \n\nPlease note that this is only a hypothetical check—no refund has been approved, issued, or processed. To begin an actual return, you would need to contact support to submit a return request.',
  'extras': {'signature': 'EosDCogDAWkUfRNaLQl3C8sm9HqKoX+SitFwXc+qTQgu82YtSmrEgNRn/7AvgBH6HknTmqgyplzTgrrfHYJuvNHchbE7C11DXGZGt+rhGKtAQUU37CVj0vqQ+T+rhq8UNSlCDkLsS1XCA9zi9FHU763PS2LzJMDzNPDgy0yTiZLwPeFir0WPerQkcZm4o30zJcEqPolpDn4+g/4O56KIBUYB41hsppACAV3HMqhLCmQ9JjZjAwEk6WaeRekfGBcsS0fxha00vQUxOLOUhyllSyeZNaS3VikJCuq2+CVstfS78NDCuKnmlifxxKGKg5Uk6deqdZ+ebUBDpykK9TtabqFmqvl7r7NdoKX3C/BfKX0SRMl/HjiZ0Zay71CXFiKnXaIKdJhjwN+fDG3lJbpInsix22jYBVPmTY5Wq1X7qioW1oYaLtfwpti1Y2kAlSggD+LEjvqZraWm1LfThX4v/zVZ3LqSqV9BifAsUWpzZZFWp3TSNgn4xmimU7qP0zVnuh1fkDJPGt2o5JETzVE='}}]

## Section 9 — Multi-Turn Conversation Memory Demonstration

In [37]:
memory_thread = str(uuid.uuid4())
ask_main("I need a laptop for school.", memory_thread)
ask_main("My budget is $1,500.", memory_thread)
ask_main("I prefer something lightweight. Which one would you recommend?", memory_thread)

USER: I need a laptop for school.
BOT: [{'type': 'text', 'text': 'Hello! I would be happy to help you find the right laptop for school.\n\nHere are two great options that offer an ideal combination of portability, performance, and value for students:\n\n---\n\n### 1. **GammaAir X** — **$1,399**\n* **Specs:** AMD Ryzen 7 processor, 32GB DDR4 RAM, 512GB NVMe SSD.\n* **Why it fits:** \n  * Its **thin and light form factor** makes it easy to slip into a backpack and carry around campus all day.\n  * With **32GB of RAM**, it handles heavy multitasking with ease—perfect if you keep dozens of browser tabs, PDFs, and documents open simultaneously while studying or conducting research.\n\n---\n\n### 2. **AlphaBook Pro** — **$1,499**\n* **Specs:** 12th Gen Intel Core i7 processor, 16GB DDR4 RAM, 1TB SSD.\n* **Why it fits:**\n  * It is a **sleek ultrabook** offering a balanced blend of portability and performance.\n  * The large **1TB SSD** provides ample storage for all your coursework, presenta

[{'type': 'text',
  'text': 'Since lightweight portability is your top priority, I strongly recommend the **GammaAir X**:\n\n### **Top Recommendation: GammaAir X** — **$1,399**\n* **Why it’s the best choice for you:**\n  * **Thin & Light Form Factor:** The GammaAir X is specifically engineered with a thin, lightweight body, making it the easiest and most comfortable option to carry around in your backpack all day between classes, the library, or study sessions.\n  * **Superior Multitasking:** It packs **32GB of DDR4 RAM** alongside an **AMD Ryzen 7 processor**, ensuring buttery-smooth performance even when you have multiple apps, heavy documents, and countless browser tabs open.\n  * **Under Budget:** At **$1,399**, it comes in **$101 below your $1,500 limit**, leaving you room to pick up a protective laptop sleeve or other school supplies.\n  * **Storage:** It includes a fast **512GB NVMe SSD**, which provides plenty of space for school documents, textbooks, and software.\n\n---\n\n*(

## Section 10 — Final Integrated Chatbot Demo

In [33]:
demo_thread = str(uuid.uuid4())
ask_main("What laptops do you have for students?", demo_thread)
ask_main(f"Can you look up order {example_order_id}?", demo_thread)
ask_main("What is your refund policy?", demo_thread)

USER: What laptops do you have for students?
BOT: [{'type': 'text', 'text': "Here are two great laptop options well-suited for students, depending on your coursework and daily needs:\n\n---\n\n### 1. **GammaAir X** — **$1,399**\n* **Key Specs:** AMD Ryzen 7 processor, 32GB DDR4 RAM, 512GB NVMe SSD\n* **Why it’s great for students:** \n  * **Portability:** Features a thin and light form factor, making it easy to slip into a backpack and carry across campus.\n  * **Multitasking Power:** With 32GB of RAM and a fast Ryzen 7 processor, you can smoothly run multiple apps, research tabs, and demanding software without slowdowns.\n  * **Value:** It is the most budget-friendly option in our current lineup while delivering top-tier performance.\n\n---\n\n### 2. **NanoEdge Flex** — **$1,699**\n* **Key Specs:** Apple M1 Pro chip, 16GB Unified Memory, 512GB SSD\n* **Why it’s great for students:**\n  * **Versatility:** Designed as a 2-in-1 convertible, allowing you to easily switch between laptop mo

[{'type': 'text',
  'text': 'Here is our sample refund policy:\n\n* **Return Window:** Returns are accepted within **30 days** of delivery.\n* **Condition:** The laptop must be in **good condition**.\n* **Proof of Purchase:** A valid receipt or proof of purchase is required.\n* **Refund Method:** Refunds are issued to the original payment method.\n* **How to Request:** Contact customer support to initiate a return request.\n\n*(Note: These are sample policies for demonstration purposes.)* \n\nIf you would like to check refund eligibility for a specific order, please let me know!',
  'extras': {'signature': 'EvUECvIEAWkUfRMDQGuo0GyDJqC7KexH80wNOLhb+bv90LG0dCHk5hWJWbAomTze0HxMXvVqS+570usvlVQnv39tTO9VYwpGpoN6PjI2H9xPXmmVn/oLLJ5dAOJ9/MWb2Ni+VWLxcmtHujAOTc9KdeHx6YNGdZa+lI91tWN9uFAK8EcTFgxS3gf/H0biCaJnv/E1rNzXmdnmN92WyETqgO9TX0IufqwPPaYoGe3G3p+LoQyK2pdjp7OASwRqrDLGz0anfjPAhZ4ZRBkAM8rL7JtegDjFneQ+gBrXXvQIuS7UHnJR/s+u6VlC1Nc88LZj56jkxMktFPZ2y0773bv6W1x16Dt3dCXaCbzYopzlZUngpReYuvtjfDx1+afW46snW

In [34]:
# Optional: ask the same chatbot multiple questions with preserved conversation history.
interactive_thread = str(uuid.uuid4())
print("Type 'exit' to stop.")
while True:
    question = input("Customer: ").strip()
    if question.lower() in {"exit", "quit"}:
        break
    if question:
        ask_main(question, interactive_thread)

Type 'exit' to stop.
Customer: exit
